# FleetGuard — Entraînement du modèle de remplacement de l'Agent LLM

Ce notebook reprend l'ensemble du pipeline utilisé :
1. Chargement du dataset corrigé
2. Correction de la fuite de target (`anomaly_count >= 2 -> CRITIQUE` de façon déterministe)
3. Préparation X / y (encodage One-Hot)
4. Split train/test stratifié 80/20
5. Entraînement et comparaison de 3 modèles (Decision Tree, Random Forest, XGBoost)
6. Choix du modèle final (Random Forest) et évaluation détaillée
7. Ré-entraînement sur 100% des données et export du modèle final (`.pkl`)

Dataset de départ : `FleetGuard_ML_dataset_5000.csv` (5000 lignes, 19 features + target `severity`).

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

RANDOM_STATE = 42

## 1. Chargement du dataset original

On part du dataset brut de 5000 lignes tel que fourni.

In [2]:
df = pd.read_csv('FleetGuard_ML_dataset_5000.csv')
print('Shape:', df.shape)
df.head()

Shape: (5000, 20)


,trip_distance_km,trip_duration_hours,reference_consumption_l_100km,calculated_actual_consumption_l_100km,deviation_percent,fuel_difference_l,overconsumption_percent,idle_ratio_percent,fuel_added_l,refuel_ratio_percent,fuel_start_percent,fuel_end_percent,vehicle_type,fuel_type,mission_status,overconsumption_detected,excessive_idle_detected,impossible_refuel_detected,anomaly_count,severity
0,194.28,7.28,5.0,5.52,10.34,1.00,10.34,28.34,3.90,8.68,67.26,52.11,Sedan,Diesel,PARTIAL,1,0,0,1,2
1,33.00,0.75,7.5,6.75,-9.98,-0.25,0.00,19.71,14.78,24.63,48.31,69.23,SUV,Diesel,COMPLETED,0,0,0,0,1
2,139.80,2.46,8.5,8.62,1.41,0.17,1.41,25.21,4.94,7.59,46.20,35.26,SUV,Essence,COMPLETED,0,0,0,0,1
3,300.09,12.00,8.2,8.86,7.99,1.97,7.99,32.74,4.51,6.44,56.09,24.57,Truck,Diesel,PARTIAL,0,1,0,1,2
4,50.82,0.59,5.0,7.00,40.01,1.02,40.01,46.16,60.30,133.99,62.74,188.83,Sedan,Diesel,COMPLETED,1,1,1,3,3


## 2. Audit rapide (rappel)

L'audit qualité complet (intégrité, cohérence mathématique, cohérence métier, répartition des anomalies)
a été fait en amont. Le seul problème identifié :

**`anomaly_count >= 2` donnait `severity = 3` (CRITIQUE) de façon 100% déterministe sur 1205 lignes (24,1% du dataset).**

C'est un raccourci que le modèle apprendrait trivialement (une seule coupure), au lieu de pondérer les
variables continues comme voulu. On corrige ci-dessous.

In [3]:
computed = df['overconsumption_detected'] + df['excessive_idle_detected'] + df['impossible_refuel_detected']
assert (computed == df['anomaly_count']).all(), "anomaly_count incoherent avec les 3 flags"

ct = pd.crosstab(df['anomaly_count'], df['severity'])
print("Crosstab AVANT correction (anomaly_count x severity) :")
print(ct)

Crosstab AVANT correction (anomaly_count x severity) :
severity          1     2    3
anomaly_count                 
0              1671    83    0
1                95  1566  380
2                 0     0  790
3                 0     0  415


## 3. Correction de la fuite de target

Pour chaque groupe `anomaly_count == 2` et `anomaly_count == 3`, on calcule un score d'intensité
(dépassement au-dessus du seuil de chaque anomalie active), et on bascule les **8% les plus faibles
en intensité** de CRITIQUE vers SURVEILLANCE. Cela casse la règle déterministe tout en gardant
la tendance métier (plusieurs anomalies simultanées restent très majoritairement critiques).

In [4]:
np.random.seed(RANDOM_STATE)

mask_multi = df['anomaly_count'] >= 2
sub_idx = df[mask_multi].index

oc_excess = np.where(df.loc[sub_idx, 'overconsumption_detected'] == 1,
                      (df.loc[sub_idx, 'deviation_percent'] - 10).clip(lower=0), 0)
ei_excess = np.where(df.loc[sub_idx, 'excessive_idle_detected'] == 1,
                      (df.loc[sub_idx, 'idle_ratio_percent'] - 30).clip(lower=0), 0)
ir_excess = np.where(df.loc[sub_idx, 'impossible_refuel_detected'] == 1,
                      (df.loc[sub_idx, 'refuel_ratio_percent'] - 100).clip(lower=0), 0)

def norm(x):
    x = np.array(x, dtype=float)
    return x / x.max() if x.max() > 0 else x

intensity = pd.Series(norm(oc_excess) + norm(ei_excess) + norm(ir_excess), index=sub_idx)

PCT = 0.08
flip_indices = []
for ac_val, group in df.loc[sub_idx].groupby('anomaly_count'):
    grp_intensity = intensity.loc[group.index].sort_values()
    n_flip = int(round(len(group) * PCT))
    flip_indices.extend(grp_intensity.index[:n_flip])
    print(f'anomaly_count={ac_val}: {len(group)} lignes -> {n_flip} basculees ({n_flip/len(group)*100:.1f}%)')

df_corrected = df.copy()
df_corrected.loc[flip_indices, 'severity'] = 2

print('\nTotal lignes basculees:', len(flip_indices))
df_corrected.to_csv('FleetGuard_ML_dataset_5000_corrected.csv', index=False)

anomaly_count=2: 790 lignes -> 63 basculees (8.0%)
anomaly_count=3: 415 lignes -> 33 basculees (8.0%)

Total lignes basculees: 96


In [5]:
print("Repartition severity APRES correction :")
print(df_corrected['severity'].value_counts(normalize=True).sort_index().round(4) * 100)

ct2 = pd.crosstab(df_corrected['anomaly_count'], df_corrected['severity'])
print("\nCrosstab APRES correction :")
print(ct2)
print((ct2.div(ct2.sum(axis=1), axis=0) * 100).round(1))

Repartition severity APRES correction :
severity
1    35.32
2    34.90
3    29.78
Name: proportion, dtype: float64

Crosstab APRES correction :
severity          1     2    3
anomaly_count                 
0              1671    83    0
1                95  1566  380
2                 0    63  727
3                 0    33  382
severity          1     2     3
anomaly_count                  
0              95.3   4.7   0.0
1               4.7  76.7  18.6
2               0.0   8.0  92.0
3               0.0   8.0  92.0


## 4. Préparation X / y

- Target : `severity`
- Encodage One-Hot pour les variables catégorielles (`vehicle_type`, `fuel_type`, `mission_status`) —
  adapté aux modèles à base d'arbres, pas d'ordre artificiel introduit entre les catégories.

In [6]:
y = df_corrected['severity']
X = df_corrected.drop(columns=['severity'])

cat_cols = ['vehicle_type', 'fuel_type', 'mission_status']
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=False)
FEATURE_COLUMNS = list(X_encoded.columns)

print('Shape X apres encodage:', X_encoded.shape)
print('Colonnes:', FEATURE_COLUMNS)

Shape X apres encodage: (5000, 25)
Colonnes: ['trip_distance_km', 'trip_duration_hours', 'reference_consumption_l_100km', 'calculated_actual_consumption_l_100km', 'deviation_percent', 'fuel_difference_l', 'overconsumption_percent', 'idle_ratio_percent', 'fuel_added_l', 'refuel_ratio_percent', 'fuel_start_percent', 'fuel_end_percent', 'overconsumption_detected', 'excessive_idle_detected', 'impossible_refuel_detected', 'anomaly_count', 'vehicle_type_Pickup', 'vehicle_type_SUV', 'vehicle_type_Sedan', 'vehicle_type_Truck', 'vehicle_type_Van', 'fuel_type_Diesel', 'fuel_type_Essence', 'mission_status_COMPLETED', 'mission_status_PARTIAL']


## 5. Split train/test (80/20, stratifié sur `severity`)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('\nRepartition train (%):')
print(y_train.value_counts(normalize=True).sort_index().round(3) * 100)
print('\nRepartition test (%):')
print(y_test.value_counts(normalize=True).sort_index().round(3) * 100)

Train: (4000, 25)  Test: (1000, 25)

Repartition train (%):
severity
1    35.3
2    34.9
3    29.8
Name: proportion, dtype: float64

Repartition test (%):
severity
1    35.3
2    34.9
3    29.8
Name: proportion, dtype: float64


## 6. Entraînement et comparaison de 3 modèles

- Decision Tree (max_depth=8)
- Random Forest (n_estimators=200, max_depth=10)
- XGBoost (n_estimators=200, max_depth=5, learning_rate=0.1)

Métriques : accuracy globale + F1-score macro (plus pertinent ici car on veut bien détecter
les CRITIQUE, pas juste avoir un bon score global).

In [8]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                              random_state=RANDOM_STATE, eval_metric='mlogloss'),
}

# XGBoost attend des labels commencant a 0
y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

results = {}
for name, model in models.items():
    if name == 'XGBoost':
        model.fit(X_train, y_train_xgb)
        preds = model.predict(X_test) + 1
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1m = f1_score(y_test, preds, average='macro')
    results[name] = {'model': model, 'preds': preds, 'acc': acc, 'f1_macro': f1m}

    print(f'=== {name} ===')
    print(f'Accuracy: {acc:.4f}  |  F1-macro: {f1m:.4f}')
    print(classification_report(y_test, preds, digits=3))
    print()

=== Decision Tree ===
Accuracy: 0.8510  |  F1-macro: 0.8521
              precision    recall  f1-score   support

           1      0.957     0.878     0.916       353
           2      0.758     0.845     0.799       349
           3      0.857     0.826     0.841       298

    accuracy                          0.851      1000
   macro avg      0.857     0.850     0.852      1000
weighted avg      0.858     0.851     0.853      1000




=== Random Forest ===
Accuracy: 0.8730  |  F1-macro: 0.8729
              precision    recall  f1-score   support

           1      0.956     0.918     0.936       353
           2      0.782     0.883     0.829       349
           3      0.903     0.809     0.853       298

    accuracy                          0.873      1000
   macro avg      0.880     0.870     0.873      1000
weighted avg      0.879     0.873     0.874      1000




=== XGBoost ===
Accuracy: 0.8640  |  F1-macro: 0.8640
              precision    recall  f1-score   support

           1      0.955     0.909     0.932       353
           2      0.768     0.874     0.818       349
           3      0.891     0.799     0.842       298

    accuracy                          0.864      1000
   macro avg      0.872     0.861     0.864      1000
weighted avg      0.871     0.864     0.865      1000




In [9]:
print('=== Resume comparatif ===')
comp = pd.DataFrame({
    name: {'accuracy': r['acc'], 'f1_macro': r['f1_macro']}
    for name, r in results.items()
}).T
comp

=== Resume comparatif ===


,accuracy,f1_macro
Decision Tree,0.851,0.852096
Random Forest,0.873,0.872862
XGBoost,0.864,0.863986


## 7. Choix du modèle final : Random Forest

Random Forest obtient le meilleur score sur les deux métriques, tout en restant simple à intégrer
(pas besoin de la complexité supplémentaire de XGBoost ici).

In [10]:
best_preds = results['Random Forest']['preds']
cm = confusion_matrix(y_test, best_preds)
print('Matrice de confusion (lignes=vrai, colonnes=predit) [1=NORMAL, 2=SURVEILLANCE, 3=CRITIQUE]:')
print(cm)

Matrice de confusion (lignes=vrai, colonnes=predit) [1=NORMAL, 2=SURVEILLANCE, 3=CRITIQUE]:
[[324  29   0]
 [ 15 308  26]
 [  0  57 241]]


In [11]:
importances = pd.Series(
    results['Random Forest']['model'].feature_importances_, index=X_encoded.columns
).sort_values(ascending=False)

print('Importance des variables (Random Forest):')
importances.head(15)

Importance des variables (Random Forest):


anomaly_count                            0.302432
idle_ratio_percent                       0.121214
deviation_percent                        0.099199
overconsumption_percent                  0.098375
excessive_idle_detected                  0.091641
refuel_ratio_percent                     0.047915
overconsumption_detected                 0.043763
fuel_end_percent                         0.035737
fuel_added_l                             0.032158
fuel_difference_l                        0.026147
impossible_refuel_detected               0.024816
calculated_actual_consumption_l_100km    0.016667
trip_distance_km                         0.014478
trip_duration_hours                      0.013795
fuel_start_percent                       0.013749
dtype: float64

## 8. Ré-entraînement sur 100% des données et export

Pour la mise en production, on ré-entraîne le modèle final sur l'intégralité du dataset disponible
(train + test) afin de ne pas gâcher de données, puisque l'évaluation ci-dessus a déjà validé
la performance sur le holdout.

In [12]:
rf_final = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE)
rf_final.fit(X_encoded, y)

with open('fleetguard_severity_model.pkl', 'wb') as f:
    pickle.dump({
        'model': rf_final,
        'feature_columns': FEATURE_COLUMNS,
        'categorical_columns': cat_cols,
        'raw_feature_order': list(X.columns),
    }, f)

print('Modele final exporte -> fleetguard_severity_model.pkl')
print('Nombre de colonnes attendues par le modele:', len(FEATURE_COLUMNS))

Modele final exporte -> fleetguard_severity_model.pkl
Nombre de colonnes attendues par le modele: 25


## 9. Test rapide sur l'exemple réel du projet

On vérifie que le modèle retrouve bien `severity = 3` sur l'exemple de trajet Casablanca → Rabat
fourni au départ (surconsommation de 48,38%).

In [13]:
example_features = {
    'trip_distance_km': 87, 'trip_duration_hours': 4.0,
    'reference_consumption_l_100km': 5.5, 'calculated_actual_consumption_l_100km': 8.16,
    'deviation_percent': 48.38, 'fuel_difference_l': 2.32, 'overconsumption_percent': 48.38,
    'idle_ratio_percent': 0, 'fuel_added_l': 0, 'refuel_ratio_percent': 0,
    'fuel_start_percent': 83.8, 'fuel_end_percent': 69.6,
    'vehicle_type': 'Van', 'fuel_type': 'Diesel', 'mission_status': 'COMPLETED',
    'overconsumption_detected': 1, 'excessive_idle_detected': 0,
    'impossible_refuel_detected': 0, 'anomaly_count': 1,
}

row = pd.DataFrame([example_features])
row = pd.get_dummies(row, columns=cat_cols, drop_first=False)
row = row.reindex(columns=FEATURE_COLUMNS, fill_value=0)

pred = rf_final.predict(row)[0]
print(f"Severity predite : {pred} (attendu : 3 - CRITIQUE)")

Severity predite : 3 (attendu : 3 - CRITIQUE)
